In [33]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel ,CLIPTokenizer
import json
import torch.nn as nn

In [58]:
# Load JSON
with open("data.json", "r") as file:
    data = json.load(file)

In [24]:
# Initialize CLIP tokenizer
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

/home/amerti/anaconda3/envs/torch/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [29]:
# Load CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/home/amerti/anaconda3/envs/torch/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [61]:
import json
from transformers import CLIPTokenizer

# Load JSON file
with open("data.json", "r") as file:
    data = json.load(file)

# Initialize CLIP tokenizer
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

# Preprocess data
def preprocess_data(data):
    processed_data = []
    for item in data:
        image_path = item["image_path"]
        for qa_pair in item["qa"]:
            question = qa_pair["question"]
            answer = qa_pair["answer"]

            # Tokenize question and answer
            question_tokens = tokenizer(question, return_tensors="pt", padding="max_length", max_length=32, truncation=True)
            answer_tokens = tokenizer(answer, return_tensors="pt", padding="max_length", max_length=32, truncation=True)

            processed_data.append({
                "image_path": image_path,
                "question_tokens": question_tokens,
                "answer_tokens": answer_tokens
            })
    return processed_data

processed_data = preprocess_data(data)


In [53]:
def extract_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        image_embedding = clip_model.get_image_features(**inputs)  # Shape: [1, 512]
    return image_embedding

In [54]:
def create_combined_embedding(image_path, question_tokens):
    # Extract image embedding
    image_embedding = extract_image_embedding(image_path)

    # Flatten question tokens for input
    question_embedding = clip_model.get_text_features(**question_tokens)

    # Combine embeddings (concatenate or use other fusion methods)
    combined_embedding = torch.cat([image_embedding, question_embedding], dim=1)  # Shape: [1, 512 + 512]
    return combined_embedding

In [55]:
class ClipQAModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size):
        super(ClipQAModel, self).__init__()
        self.fc1 = nn.Linear(embedding_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, image_embedding, question_embedding):
        combined = torch.cat([image_embedding, question_embedding], dim=1)
        hidden = self.relu(self.fc1(combined))
        logits = self.fc2(hidden)
        return logits


In [62]:
# Initialize model, loss, and optimizer
embedding_dim = 512  # CLIP embedding size
hidden_dim = 256
vocab_size = tokenizer.vocab_size  # Size of the tokenizer vocabulary
qa_model = ClipQAModel(embedding_dim, hidden_dim, vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(qa_model.parameters(), lr=1e-4)

# Training loop for small dataset
num_epochs = 10
qa_model.train()

for epoch in range(num_epochs):
    for sample in processed_data:
        image_path = sample["image_path"]
        question_tokens = sample["question_tokens"]
        answer_tokens = sample["answer_tokens"]

        # Get combined embedding
        combined_embedding = create_combined_embedding(image_path, question_tokens)

        # Forward pass
        logits = qa_model(combined_embedding[:, :512], combined_embedding[:, 512:])

        # The target must match the logits batch size
        target = answer_tokens["input_ids"].squeeze(0)  # Shape: [seq_len]
        target = target[:logits.shape[0]]  # Ensure length match for padding

        # Compute loss
        loss = criterion(logits, target)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")


Epoch 1, Loss: 10.176156997680664
Epoch 2, Loss: 9.334063529968262
Epoch 3, Loss: 8.313776969909668
Epoch 4, Loss: 6.9728803634643555
Epoch 5, Loss: 5.255088806152344
Epoch 6, Loss: 3.154733657836914
Epoch 7, Loss: 1.1823291778564453
Epoch 8, Loss: 0.37972626090049744
Epoch 9, Loss: 0.19157744944095612
Epoch 10, Loss: 0.12793679535388947


In [69]:
def predict_answer(image_path, question):
    question_tokens = tokenizer(question, return_tensors="pt", padding="max_length", max_length=32, truncation=True)
    combined_embedding = create_combined_embedding(image_path, question_tokens)

    with torch.no_grad():
        logits = qa_model(combined_embedding[:, :512], combined_embedding[:, 512:])
        predicted_ids = logits.argmax(dim=-1)  # Get the index of the highest logit
        predicted_answer = tokenizer.decode(predicted_ids, skip_special_tokens=True)  # Skip special tokens

    return predicted_answer

# Example usage
predicted = predict_answer("images/dog.jpg", "why is the color of the cat?")
print(f"Predicted Answer: {predicted}")


Predicted Answer: 


In [70]:
print(tokenizer.decode(answer_tokens["input_ids"][0], skip_special_tokens=False))


<|startoftext|>on the grass <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [50]:
print("Logits shape:", logits.shape)
print("Predicted tokens:", tokenizer.decode(predicted))


Logits shape: torch.Size([1, 49408])
Predicted tokens: 


In [71]:
# Debugging the predicted tokens
predicted_ids = logits.argmax(dim=-1).squeeze(0)  # Shape: [seq_len]
print("Predicted IDs:", predicted_ids.tolist())  # Check the raw predicted IDs

# Decode without skipping special tokens
predicted_answer = tokenizer.decode(predicted_ids, skip_special_tokens=False)
print("Decoded Answer (with special tokens):", predicted_answer)

# Decode while skipping special tokens
predicted_answer = tokenizer.decode(predicted_ids, skip_special_tokens=True)
print("Decoded Answer (clean):", predicted_answer)


Predicted IDs: 49406
Decoded Answer (with special tokens): <|startoftext|>
Decoded Answer (clean): <|startoftext|>
